**Plant Disease Detection — Transfer Learning (ResNet50)**

**Objective**
Improve on the Sequential ConvNet baseline (66% val accuracy) using
transfer learning with ResNet50 pretrained on ImageNet.

**Dataset**
<br>
** Plant Diseases Dataset** from Kaggle
- 70,295 training images, 17,572 validation images
- 38 classes covering healthy and diseased leaves across crops
  including tomato, apple, corn, grape, potato and more
- Color JPG images resized to 224x224 for ResNet50 compatibility

**Approach**
- Use ResNet50 pretrained on ImageNet as the base model
- 3-stage gradual unfreezing to preserve pretrained weights
- Stage 1: train head only (base frozen)
- Stage 2: unfreeze last 30 layers with small learning rate
- Stage 3: unfreeze entire model with very small learning rate
- Save checkpoints to Google Drive after every improvement

**Learning Objectives**
- Why transfer learning outperforms training from scratch
- How gradual unfreezing works and why the learning rate must
  decrease at each stage
- How pretrained ImageNet features transfer to plant disease detection
- How to build a transfer learning model using the Keras Functional API






In [4]:
#Dataset Download
import os

if not os.path.exists('New Plant Diseases Dataset(Augmented)'):
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d vipoooool/new-plant-diseases-dataset
    !unzip -q new-plant-diseases-dataset.zip
    print('Download complete!')
else:
    print('Dataset already exists — skipping download')

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.70G/2.70G [00:13<00:00, 214MB/s]

Download complete!


In [5]:
#Verifying Dataset Structure and Components
import os

TRAIN_DIR = 'New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
VAL_DIR   = 'New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid'

CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)

print(f'Number of classes: {NUM_CLASSES}')
print(f'Total train images: {sum(len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in CLASS_NAMES)}')
print(f'Total val images:   {sum(len(os.listdir(os.path.join(VAL_DIR, c))) for c in CLASS_NAMES)}')

Number of classes: 38
Total train images: 70295
Total val images:   17572


In [6]:
# Libraries Imports

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
#Data Pipeline

# ResNet50 expects 224x224 — standard ImageNet size
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32   # smaller batch — larger images use more GPU memory
NUM_CLASSES = len(CLASS_NAMES)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

print('Training batches:  ', len(train_data))
print('Validation batches:', len(val_data))

Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.
Training batches:   2197
Validation batches: 550


In [8]:
#Building the model

tf.keras.backend.clear_session()

# Load ResNet50 with ImageNet weights, no top classifier
base_model = ResNet50(
    weights='imagenet',
    include_top=False,        # removes ResNet's original 1000-class head
    input_shape=(224, 224, 3)
)

# Stage 1: freeze entire base
base_model.trainable = False

# Build new head for 38 plant disease classes
inputs  = Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs, name='ResNet50_PlantDisease')

print(f'Total layers: {len(model.layers)}')
print(f'Trainable layers: {len([l for l in model.layers if l.trainable])}')
print(f'Total parameters:     {model.count_params():,}')

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total layers: 7
Trainable layers: 6
Total parameters:     24,130,214


In [9]:
#Stage 1 - Training Heads, Base Frozen

# Use normal learning rate since head weights are random

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_s1 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

print('Stage 1: Training head only (base frozen)')
print(f'Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

history_s1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks_s1,
    verbose=1
)

print('\nStage 1 complete!')
s1_acc = max(history_s1.history['val_accuracy'])
print(f'Best val accuracy: {s1_acc:.4f} ({s1_acc*100:.1f}%)')

Stage 1: Training head only (base frozen)
Trainable params: 538,406
Epoch 1/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 468s 208ms/step - accuracy: 0.4480 - loss: 1.8930 - val_accuracy: 0.5744 - val_loss: 1.4376
Epoch 2/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 466s 212ms/step - accuracy: 0.5666 - loss: 1.4310 - val_accuracy: 0.5989 - val_loss: 1.3320
Epoch 3/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 470s 214ms/step - accuracy: 0.6006 - loss: 1.2996 - val_accuracy: 0.6154 - val_loss: 1.2903
Epoch 4/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 470s 214ms/step - accuracy: 0.6192 - loss: 1.2417 - val_accuracy: 0.6464 - val_loss: 1.1906
Epoch 5/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 472s 215ms/step - accuracy: 0.6315 - loss: 1.1997 - val_accuracy: 0.6580 - val_loss: 1.1613
Epoch 6/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 439s 200ms/step - accuracy: 0.6395 - loss: 1.1619 - val_accuracy: 0.6657 - val_loss: 1.1197
Epoch 7/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 446s 203ms/step - accuracy: 0.6472 - loss: 1.1384 - val_accuracy: 0.6600 - val_loss: 1

In [10]:
import os

CHECKPOINT_PATH = 'stage2_checkpoint.keras'

# If a checkpoint exists, load it and resume
if os.path.exists(CHECKPOINT_PATH):
    print('Checkpoint found — loading and resuming...')
    model.load_weights(CHECKPOINT_PATH)
else:
    print('No checkpoint found — starting Stage 2 fresh')

# Unfreeze last 30 layers
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

from tensorflow.keras.callbacks import ModelCheckpoint

callbacks_s2 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,    # only saves when val_accuracy improves
        save_weights_only=False,
        verbose=1
    )
]

print('Stage 2: Unfreezing last 30 layers')
print(f'Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

history_s2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks_s2,
    verbose=1
)

print('\nStage 2 complete!')
s2_acc = max(history_s2.history['val_accuracy'])
print(f'Best val accuracy: {s2_acc:.4f} ({s2_acc*100:.1f}%)')

No checkpoint found — starting Stage 2 fresh
Stage 2: Unfreezing last 30 layers
Trainable params: 14,988,582
Epoch 1/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step - accuracy: 0.3654 - loss: 9.6518
Epoch 1: val_accuracy improved from None to 0.40565, saving model to stage2_checkpoint.keras

Epoch 1: finished saving model to stage2_checkpoint.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 462s 205ms/step - accuracy: 0.4485 - loss: 5.7100 - val_accuracy: 0.4056 - val_loss: 5.9761 - learning_rate: 1.0000e-05
Epoch 2/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.5432 - loss: 2.2734
Epoch 2: val_accuracy improved from 0.40565 to 0.52692, saving model to stage2_checkpoint.keras

Epoch 2: finished saving model to stage2_checkpoint.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 451s 205ms/step - accuracy: 0.5571 - loss: 2.0058 - val_accuracy: 0.5269 - val_loss: 2.2730 - learning_rate: 1.0000e-05
Epoch 3/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.5987 - loss: 1.4799
Epoch 3: val

In [11]:
CHECKPOINT_PATH_S3 = 'stage3_checkpoint.keras'

if os.path.exists(CHECKPOINT_PATH_S3):
    print('Stage 3 checkpoint found — loading and resuming...')
    model.load_weights(CHECKPOINT_PATH_S3)
else:
    print('No checkpoint — starting Stage 3 fresh')

# Unfreeze everything
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_s3 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    ModelCheckpoint(
        CHECKPOINT_PATH_S3,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    )
]

print('Stage 3: Full model unfrozen')
print(f'Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

history_s3 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks_s3,
    verbose=1
)

print('\nStage 3 complete!')
s3_acc = max(history_s3.history['val_accuracy'])
print(f'Best val accuracy: {s3_acc:.4f} ({s3_acc*100:.1f}%)')

No checkpoint — starting Stage 3 fresh
Stage 3: Full model unfrozen
Trainable params: 24,072,998
Epoch 1/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - accuracy: 0.1149 - loss: 30.6032
Epoch 1: val_accuracy improved from None to 0.42590, saving model to stage3_checkpoint.keras

Epoch 1: finished saving model to stage3_checkpoint.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 511s 216ms/step - accuracy: 0.2016 - loss: 21.6937 - val_accuracy: 0.4259 - val_loss: 8.9617 - learning_rate: 1.0000e-06
Epoch 2/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.4199 - loss: 9.3017
Epoch 2: val_accuracy improved from 0.42590 to 0.65064, saving model to stage3_checkpoint.keras

Epoch 2: finished saving model to stage3_checkpoint.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 449s 204ms/step - accuracy: 0.4665 - loss: 7.9570 - val_accuracy: 0.6506 - val_loss: 3.9195 - learning_rate: 1.0000e-06
Epoch 3/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.5788 - loss: 5.2544
Epoch 3: val_accuracy 

##**Results & Conclusion**

**Final Model Performance**

| Metric | Value |
|---|---|
| Validation Accuracy | 92.3% |
| Validation Loss | 0.3925 |
| Number of Classes | 38 |
| Total Epochs | 30 (3 stages x 10) |
| Best Checkpoint | Stage 3, Epoch 10 |

<br>

**Model Comparison**
| Model | Val Accuracy | Epochs | Starting Weights |
|---|---|---|---|
| ConvNet from scratch | 66% | 10 | Random |
| ResNet50 Transfer Learning | **91.9%** | 30 | ImageNet (1.4M images) |
| Improvement | **+25.9%** | - | - |

<br>

**Why Transfer Learning is Better**

ResNet50 pretrained on 1.4 million ImageNet images already understood
edges, textures, shapes and colour patterns. Plant disease recognition
relies on exactly these features: leaf textures, colour changes, spot
patterns and lesion boundaries. The pretrained weights gave the model
a head start that a scratch ConvNet cannot match with 54,000 images alone.

<br>

**The 3-Stage Unfreezing Strategy**

| Stage | Layers Trained | Learning Rate | Best Val Accuracy |
|---|---|---|---|
| Stage 1 | Head only (538K params) | 1e-3 | ~60% |
| Stage 2 | Head + last 30 ResNet layers | 1e-5 | ~88% |
| Stage 3 | Entire model (25M params) | 1e-6 | **91.9%** |

<br>
<br>


**Key Learnings**
- Transfer learning delivered **+25.9% accuracy** over training from scratch
- Gradual unfreezing is critical: unfreezing everything at once
  with a high learning rate destroys pretrained weights
- The learning rate must decrease at each stage as more layers unfreeze
- Always save model checkpoints to persistent storage: local Colab
  storage is wiped on every session restart
- ImageNet features transfer well to biological image domains
  like plant disease detection
